# Analisis Keterlambatan Pengiriman

Notebook ini mengerjakan seluruh bagian tugas di atas satu file data: `shipments.csv`.

| Bagian | Isi |
| --- | --- |
| 1 | Memuat data, cek shape, info, nilai kosong, dan duplikat |
| 2 | Pembersihan (tipe numerik, dropna, drop_duplicates), kolom `delay_hours`, statistik NumPy |
| 3 | Grouping manual pakai loop vs `groupby`, bar chart per kurir, korelasi jarak dan berat |
| 4 | Train/test + Linear Regression, komparasi rute jauh vs dekat, satu insight bisnis |

**Dataset**: `shipments.csv`, 220 baris, 7 kolom (`tracking_number`, `courier`, `weight_kg`,
`distance_km`, `promised_hours`, `actual_hours`, `delay_hours`).

## 1. Memuat data

Semua library dipakai di seluruh notebook. `%matplotlib inline` supaya figure ikut tersimpan
di output Colab.

In [17]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

DATA_PATH = Path("shipments.csv")
if not DATA_PATH.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(
            "shipments.csv tidak ada di working directory. Unggah file dulu, lalu jalankan ulang sel ini."
        )
    unggahan = files.upload()
    DATA_PATH = Path(next(iter(unggahan)))

df = pd.read_csv(DATA_PATH)
print("file:", DATA_PATH)
print("shape:", df.shape)
df.head()

file: shipments.csv
shape: (220, 7)


,tracking_number,courier,weight_kg,distance_km,promised_hours,actual_hours,delay_hours
0,ANT0140,Sari,2.3,50.4,19,26.3,7.3
1,ANT0066,Andi,2.8,14.9,10,13.1,3.1
2,ANT0096,Budi,3.0,26.0,12,17.2,5.2
3,ANT0023,Dedi,3.1,13.6,9,10.3,1.3
4,ANT0064,Andi,2.0,11.5,9,10.8,1.8


### Kondisi awal data

`shape` memberi ukuran, `info()` memberi tipe tiap kolom, lalu jumlah nilai kosong dan baris
duplikat dihitung sebelum ada perubahan apa pun.

In [18]:
print("baris x kolom:", df.shape)
print()
df.info()

print("nilai kosong per kolom:")
print(df.isna().sum().to_string())
print()
print("total sel kosong          :", int(df.isna().sum().sum()))
print("baris duplikat penuh      :", int(df.duplicated().sum()))
print("tracking_number duplikat  :", int(df["tracking_number"].duplicated().sum()))
print("unique courier            :", sorted(df["courier"].unique()))

baris x kolom: (220, 7)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220 entries, 0 to 219
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   tracking_number  220 non-null    object 
 1   courier          220 non-null    object 
 2   weight_kg        220 non-null    float64
 3   distance_km      220 non-null    float64
 4   promised_hours   220 non-null    int64  
 5   actual_hours     220 non-null    float64
 6   delay_hours      220 non-null    float64
dtypes: float64(4), int64(1), object(2)
memory usage: 12.2+ KB
nilai kosong per kolom:
tracking_number    0
courier            0
weight_kg          0
distance_km        0
promised_hours     0
actual_hours       0
delay_hours        0

total sel kosong          : 0
baris duplikat penuh      : 0
tracking_number duplikat  : 0
unique courier            : ['Andi', 'Budi', 'Citra', 'Dedi', 'Sari']
